# Regime Window Review Validation v1

本 notebook 验证 social/X 上的 regime-aware model review 想法只能作为 `idea_only` 假设；可运行逻辑必须落到本仓库 `src.analysis` 的 regime/window 模型差异接口和 manifest/export。

In [1]:
from pathlib import Path
import json
import sys

import pandas as pd

REPO_ROOT = Path.cwd().resolve()
for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / 'AGENTS.md').exists() and (candidate / 'src').exists():
        REPO_ROOT = candidate
        break
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.analysis import (
    analyze_model_effects_by_regime_and_window,
    build_regime_window_analysis_report,
    compare_models,
    compute_regime_metrics,
    compute_window_metrics,
    write_analysis_report,
)
from src.core import DatasetSnapshot

CARD = REPO_ROOT / 'research' / 'regime_window_review_validation_v1'
CARD.mkdir(parents=True, exist_ok=True)

SOURCE = {
    'source_id': 'x:regime-aware-quant-search',
    'source_type': 'x',
    'title': 'Regime-aware quant evaluation search',
    'url': 'https://x.com/search?q=regime%20aware%20quant%20evaluation',
    'summary': (
        'Social-source idea placeholder for volatility/drawdown regime review; '
        'only usable after it becomes a reproducible src.analysis window/regime test.'
    ),
    'retrieved_at': '2026-06-19',
    'access_status': 'search_placeholder',
}
EVIDENCE_REQUIREMENTS = [
    'license_review',
    'research_notebook',
    'focused_pytest',
    'manifest_or_export',
    'chinese_conclusion',
]
SOURCE

{'source_id': 'x:regime-aware-quant-search',
 'source_type': 'x',
 'title': 'Regime-aware quant evaluation search',
 'url': 'https://x.com/search?q=regime%20aware%20quant%20evaluation',
 'summary': 'Social-source idea placeholder for volatility/drawdown regime review; only usable after it becomes a reproducible src.analysis window/regime test.',
 'retrieved_at': '2026-06-19',
 'access_status': 'search_placeholder'}

In [2]:
dates = list(pd.date_range('2026-01-01', periods=8, freq='D')) * 2
regimes = ['bull', 'bull', 'bull', 'neutral', 'stress', 'stress', 'stress', 'downtrend'] * 2
daily = pd.DataFrame(
    {
        'date': dates,
        'model': ['ridge_mse'] * 8 + ['gcn_mse'] * 8,
        'regime': regimes,
        'nav': [
            1.00, 1.02, 1.04, 1.03, 1.01, 0.99, 1.00, 1.01,
            1.00, 1.04, 1.08, 1.07, 1.02, 0.96, 0.97, 0.98,
        ],
        'benchmark_nav': [1.0, 1.01, 1.02, 1.02, 1.00, 0.99, 1.00, 1.00] * 2,
        'position': [0.2, 0.3, 0.5, 0.1, 0.4, 0.2, 0.1, 0.2] * 2,
        'turnover': [0.0, 0.1, 0.2, 0.4, 0.3, 0.1, 0.1, 0.2] * 2,
        'vix': [17, 18, 19, 24, 35, 32, 31, 22] * 2,
        'trend': [0.02, 0.03, 0.04, 0.00, -0.05, -0.06, -0.04, -0.02] * 2,
        'drawdown': [0.0, 0.0, 0.0, -0.01, -0.12, -0.14, -0.10, -0.04] * 2,
    }
)
sample_path = CARD / 'sample_daily_backtest.csv'
daily.to_csv(sample_path, index=False)
daily.head()

,date,model,regime,nav,benchmark_nav,position,turnover,vix,trend,drawdown
0,2026-01-01,ridge_mse,bull,1.00,1.00,0.2,0.0,17,0.02,0.00
1,2026-01-02,ridge_mse,bull,1.02,1.01,0.3,0.1,18,0.03,0.00
2,2026-01-03,ridge_mse,bull,1.04,1.02,0.5,0.2,19,0.04,0.00
3,2026-01-04,ridge_mse,neutral,1.03,1.02,0.1,0.4,24,0.00,-0.01
4,2026-01-05,ridge_mse,stress,1.01,1.00,0.4,0.3,35,-0.05,-0.12


In [3]:
windows = {
    'early': ('2026-01-01', '2026-01-04'),
    'stress_window': ('2026-01-05', '2026-01-07'),
}
analysis = analyze_model_effects_by_regime_and_window(
    daily,
    windows=windows,
    baseline_model='ridge_mse',
)
report = build_regime_window_analysis_report(
    artifact_id='regime_window_review_validation_v1',
    effect_analysis=analysis,
    model_comparison=compare_models(daily),
    regime_metrics=compute_regime_metrics(daily),
    window_metrics=compute_window_metrics(daily, windows=windows),
    summary={
        'status': 'regime_window_review_validation_completed',
        'source_id': SOURCE['source_id'],
        'analysis_interface': 'src.analysis.analyze_model_effects_by_regime_and_window',
        'note': 'social idea 只作为 regime/window review 假设，实际证据由 src.analysis 生成。',
    },
    metadata={
        'source_type': 'social_idea_regime_window_review',
        'source_id': SOURCE['source_id'],
        'source_used_as': 'hypothesis_only',
    },
)
dataset = DatasetSnapshot(
    name='regime_window_review_validation_sample',
    universe=('SOXL', 'SOXX'),
    start='2026-01-01',
    end='2026-01-08',
    calendar_path='data/qlib/soxl/calendars/day.txt',
    data_paths=(str(sample_path.relative_to(REPO_ROOT)),),
)
export = write_analysis_report(
    report,
    CARD,
    dataset=dataset,
    manifest_metadata={'source_type': 'social_idea_regime_window_review'},
    extra_paths={'sample_daily_backtest': 'sample_daily_backtest.csv'},
)
export

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{'schema_version': 'analysis.report.export.v1',
 'artifact_id': 'regime_window_review_validation_v1',
 'artifact_type': 'analysis_report',
 'paths': {'summary': 'analysis_summary.json',
  'model_comparison': 'model_comparison.csv',
  'regime_metrics': 'regime_metrics.csv',
  'window_metrics': 'window_metrics.csv',
  'window_regime_metrics': 'window_regime_metrics.csv',
  'qlib_return_frame': 'qlib_return_frame.csv',
  'qlib_risk_metrics': 'qlib_risk_metrics.csv',
  'segment_metrics': 'segment_metrics.csv',
  'segment_effect_matrix': 'segment_effect_matrix.csv',
  'model_effect_deltas': 'model_effect_deltas.csv',
  'model_regime_window_diagnostics': 'model_regime_window_diagnostics.csv',
  'model_difference_explanations': 'model_difference_explanations.csv',
  'model_review_briefing': 'model_review_briefing.csv',
  'failure_modes': 'failure_modes.csv',
  'sample_daily_backtest': 'sample_daily_backtest.csv',
  'manifest': 'analysis_manifest.json'},
 'table_counts': {'failure_modes': 4,
 

In [4]:
source_provenance = {
    'schema_version': 'knowledge.source_provenance.v1',
    **SOURCE,
    'source_used_as': 'hypothesis_only',
    'usage_scope': 'idea_only',
    'source_reliability': 'social_unverified',
    'external_source_checks': {
        'search_url': SOURCE['url'],
        'copied_social_post_text': False,
        'used_individual_post_as_fact': False,
        'vendored_external_code': False,
    },
    'license_note': 'idea-only；只保存搜索 URL、自有摘要和验证假设；未复制帖子正文；不作为事实来源。',
}
(CARD / 'source_provenance.json').write_text(
    json.dumps(source_provenance, ensure_ascii=False, indent=2), encoding='utf-8'
)

license_review = """# License Review

## 中文结论

- X/social 来源：https://x.com/search?q=regime%20aware%20quant%20evaluation，仅作为 idea-only 搜索占位。
- 未复制帖子正文，未保存个人帖子内容，未把社交媒体内容作为事实来源。
- 本 notebook 只验证本仓库 `src.analysis` 对 volatility/drawdown regime、window、window_regime 的模型差异分析契约。
- 可运行结果来自本仓库生成的样本 daily backtest 和 analysis manifest，不依赖外部社交媒体可访问性。
- 后续如引用具体帖子、论文或仓库，必须单独记录 URL、许可、作者和使用范围。
"""
(CARD / 'license_review.md').write_text(license_review, encoding='utf-8')

analysis_summary = json.loads((CARD / 'analysis_summary.json').read_text(encoding='utf-8'))
artifact_files = sorted(
    {
        str(path)
        for key, path in export['paths'].items()
        if key != 'manifest'
    }
)
review_summary = {
    'schema_version': 'research.regime_window_review_validation.v1',
    'source_id': SOURCE['source_id'],
    'candidate_name': 'regime_window_review',
    'target_subsystem': 'src/analysis',
    'analysis_report_schema_version': analysis_summary['schema_version'],
    'analysis_interface': 'src.analysis.analyze_model_effects_by_regime_and_window',
    'baseline_model': analysis_summary['analysis_summary']['baseline_model'],
    'top_weakness_segment': analysis_summary['analysis_overview']['top_weakness']['segment'],
    'top_strength_segment': analysis_summary['analysis_overview']['top_strength']['segment'],
    'analysis_layers': analysis_summary['analysis_layers'],
    'artifact_files': artifact_files,
    'source_used_as': 'hypothesis_only',
    'implementation_allowed': False,
    'promotion_target': 'research_only',
    'evidence_requirements': EVIDENCE_REQUIREMENTS,
}
(CARD / 'regime_window_review_summary.json').write_text(
    json.dumps(review_summary, ensure_ascii=False, indent=2), encoding='utf-8'
)
review_summary

{'schema_version': 'research.regime_window_review_validation.v1',
 'source_id': 'x:regime-aware-quant-search',
 'candidate_name': 'regime_window_review',
 'target_subsystem': 'src/analysis',
 'analysis_report_schema_version': 'analysis.report.v1',
 'analysis_interface': 'src.analysis.analyze_model_effects_by_regime_and_window',
 'baseline_model': 'ridge_mse',
 'top_weakness_segment': 'stress',
 'top_strength_segment': 'early',
 'analysis_layers': ['model_comparison',
  'regime_metrics',
  'window_metrics',
  'window_regime_metrics',
  'qlib_return_frame',
  'qlib_risk_metrics',
  'segment_metrics',
  'segment_effect_matrix',
  'model_effect_deltas',
  'model_regime_window_diagnostics',
  'model_difference_explanations',
  'model_review_briefing',
  'failure_modes'],
 'artifact_files': ['analysis_summary.json',
  'failure_modes.csv',
  'model_comparison.csv',
  'model_difference_explanations.csv',
  'model_effect_deltas.csv',
  'model_regime_window_diagnostics.csv',
  'model_review_brie

## 中文结论

- 做了什么：将 X/social 的 regime-aware review 想法固定为 `idea_only` 外部假设，并用 `src.analysis.analyze_model_effects_by_regime_and_window` 生成 all/regime/window/window_regime 的模型差异证据。
- 核心指标：`ridge_mse` 是基准；`gcn_mse` 在 `early` 强于基准，但在 `stress` 和 `stress_window::stress` 弱于基准，触发压力期脆弱画像。
- regime 差异：全区间平均不能替代 regime/window 交叉审计；同一模型在 bull/early 可以有优势，在 stress/downtrend 可能需要 block 或 rework。
- 是否晋升：否。本 notebook 只让 `regime_window_review` 通过 research evidence 门禁，不直接晋升 production，也不把社交媒体内容写入 `src` 或 skill。
- 风险：样本 daily backtest 是可复现契约样本，不代表真实交易结论；真实模型晋升需要接入完整数据、成本、滑点、objective loss 和 production readiness gate。
- 下一步：把 `model_review_briefing.csv`、`model_regime_window_diagnostics.csv` 和 `model_difference_explanations.csv` 作为 dashboard/schedule 的 review 输入，继续处理剩余 `quantbench_model_matrix` 外部知识门禁。